### ****Default Lakehouse:****`lh_Bronze_StratusCore`

> **Why this notebook exists:** When the Availability Excel sheet lands in Bronze,
> the two-row header structure collapses into a flat string table. Dataflow Gen 2
> reads column positions as fixed — applying the first header it finds to all rows.
> This caused **46% null availability values** in early testing.
>
> This notebook reads the Excel file directly (before concatenation), correctly
> maps the FM category labels from row 0, extracts date column names from row 1,
> and unpivots the wide 24-column table into a clean long-format table.
> Only then does it write to Delta.

---

### Cell 1 — Imports and Config

In [1]:
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from datetime import datetime

spark = SparkSession.builder.getOrCreate()

FILE_PATH  = "abfss://Swift@onelake.dfs.fabric.microsoft.com/lh_Bronze_StratusCoreTelecoms.Lakehouse/Files/StratusCore Telecoms.xlsx"
RUN_TS     = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

# ── Lakehouse routing ─────────────────────────────────────────────────────────
# Excel source lives in Bronze; the pre-silver staging table is written to Silver
# so the Silver notebook can read it via its own default catalog.
SILVER_CATALOG   = "lh_Silver_StratusCoreTelecoms"
PRE_SILVER_TABLE = f"{SILVER_CATALOG}.dbo.pre_silver_stratus_availability"

# Identity columns that describe the circuit — not measurements
ID_COLS = ["ID", "Segment", "Company", "Region", "BSP"]

# Column position ranges for each FM category (0-indexed, right-exclusive)
# Row 1 layout: [ID, Segment, Company, Region, BSP] [incl FM x12] [excl FM x12]
#               cols 0-4                             cols 5-16      cols 17-28
FM_CONFIG = {
    "Including_FM": (5, 17),
    "Excluding_FM": (17, 29),
}

print(f"\u2705 Pre-Silver config loaded \u2014 Run: {RUN_TS}")
print(f"   Excel source : {FILE_PATH}")
print(f"   Writing to   : {PRE_SILVER_TABLE}")


StatementMeta(, 061e5786-f0b0-4bad-8fe6-76fee4bd6201, 3, Finished, Available, Finished, False)

✅ Pre-Silver config loaded — Run: 2026-04-18 11:09:39
   Excel source : abfss://Swift@onelake.dfs.fabric.microsoft.com/lh_Bronze_StratusCoreTelecoms.Lakehouse/Files/StratusCore Telecoms.xlsx
   Writing to   : lh_Silver_StratusCoreTelecoms.dbo.pre_silver_stratus_availability


### Cell 2 — Core Extraction Function

In [2]:
def extract_availability(file_path: str) -> pd.DataFrame:
    """
    Read the Availability sheet from its Excel source — preserving the two-row
    header structure — then unpivot from wide format to long format.

    Input structure (wide):
        Row 0: [blank x5]  Including FM  [blank x11]  Excluding FM  [blank x11]
        Row 1: ID  Segment  Company  Region  BSP  Jan  Feb ... Dec  Jan  Feb ... Dec
        Row 2+: data rows (one row per circuit)

    Output structure (long):
        ID | Segment | Company | Region | BSP | Date | Availability | Identification
        One row per circuit × per month × per FM category

    Parameters
    ----------
    file_path : str
        Path to the raw Excel workbook (accessible via /lakehouse/default/Files/)

    Returns
    -------
    pd.DataFrame  long-format availability data
    """
    # ── Step 1: Read raw grid with no header interpretation ──────────────────
    df = pd.read_excel(file_path, sheet_name="Availability", header=None)
    print(f"   Raw grid shape: {df.shape}")

    # ── Step 2: Extract the two header rows separately ───────────────────────
    # Row 0: FM category labels (Including FM / Excluding FM span across columns)
    # Row 1: The actual column names we'll use (ID, Segment, ..., date values)
    category_row = df.iloc[0]   # "Including FM", "Excluding FM" labels
    header_row   = df.iloc[1]   # Column names: ID, Segment, Company, Region, BSP, dates

    print(f"   FM categories found in row 0: "
          f"{[v for v in category_row.unique() if pd.notna(v) and str(v).strip()]}")
    print(f"   Header row sample (cols 5-10): {list(header_row.iloc[5:11])}")

    # ── Step 3: Isolate data rows ─────────────────────────────────────────────
    data_rows = df.iloc[2:].reset_index(drop=True)
    data_rows.columns = header_row.values
    print(f"   Data rows: {len(data_rows)}")

    # ── Step 4: Build long-format table for each FM category ─────────────────
    all_dfs = []

    for fm_label, (start_col, end_col) in FM_CONFIG.items():
        # Extract the 12 date column names for this FM block
        date_col_names = list(header_row.iloc[start_col:end_col])

        # Build a working DataFrame: circuit identity columns + 12 date columns
        cat_df = data_rows[ID_COLS].copy()
        for i, date_col in enumerate(date_col_names):
            # Use first 10 chars of datetime string as the column name (YYYY-MM-DD)
            col_label = str(date_col)[:10]
            cat_df[col_label] = data_rows.iloc[:, start_col + i].values

        # Unpivot: wide → long
        date_str_cols = [str(d)[:10] for d in date_col_names]
        melted = pd.melt(
            cat_df,
            id_vars=ID_COLS,
            value_vars=date_str_cols,
            var_name="Date",
            value_name="Availability"
        )
        melted["Identification"] = fm_label
        all_dfs.append(melted)

        print(f"   {fm_label}: {len(melted)} rows after melt")

    # ── Step 5: Combine both FM categories ───────────────────────────────────
    result = pd.concat(all_dfs, ignore_index=True)
    print(f"\n   ✅ Final long-format shape: {result.shape}")
    print(f"   Null availability values: {result['Availability'].isna().sum()}")
    return result

StatementMeta(, 061e5786-f0b0-4bad-8fe6-76fee4bd6201, 4, Finished, Available, Finished, False)

### Cell 3 — Run Extraction

In [3]:
print("🔄 Extracting availability data from Excel source...\n")
df_avail_long = extract_availability(FILE_PATH)

print("\n📊 Preview (first 10 rows):")
print(df_avail_long.head(10).to_string(index=False))

print(f"\n📊 Identification breakdown:")
print(df_avail_long["Identification"].value_counts().to_string())

StatementMeta(, 061e5786-f0b0-4bad-8fe6-76fee4bd6201, 5, Finished, Available, Finished, False)

🔄 Extracting availability data from Excel source...

   Raw grid shape: (202, 29)
   FM categories found in row 0: ['Including FM', 'Excluding FM']
   Header row sample (cols 5-10): [datetime.datetime(2025, 1, 1, 0, 0), datetime.datetime(2025, 2, 1, 0, 0), datetime.datetime(2025, 3, 1, 0, 0), datetime.datetime(2025, 4, 1, 0, 0), datetime.datetime(2025, 5, 1, 0, 0), datetime.datetime(2025, 6, 1, 0, 0)]
   Data rows: 200
   Including_FM: 2400 rows after melt
   Excluding_FM: 2400 rows after melt

   ✅ Final long-format shape: (4800, 8)
   Null availability values: 0

📊 Preview (first 10 rows):
ID       Segment   Company  Region   BSP       Date Availability Identification
 1 Public Sector  PulseNet   North BSP-A 2025-01-01        99.86   Including_FM
 2 Public Sector   NovaTel    East BSP-D 2025-01-01        99.24   Including_FM
 3    Enterprise   NovaTel Central BSP-D 2025-01-01        99.17   Including_FM
 4 Public Sector   NovaTel    West BSP-A 2025-01-01        99.94   Including_FM
 

### Cell 4 — Add Metadata and Save to Delta

In [4]:
print("\n\U0001f4be Saving pre-silver availability table...")
print(f"   Target: {PRE_SILVER_TABLE}")

df_avail_long["_pre_silver_created_at"] = RUN_TS
df_avail_long["_source_file"]           = "StratusCore_Telecoms.xlsx"

sdf_avail = spark.createDataFrame(df_avail_long.astype(str))

spark.sql(f"DROP TABLE IF EXISTS {PRE_SILVER_TABLE}")
(
    sdf_avail
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(PRE_SILVER_TABLE)
)

row_count = spark.table(PRE_SILVER_TABLE).count()
null_count = (
    spark.table(PRE_SILVER_TABLE)
         .filter(F.col("Availability").isNull() | (F.col("Availability") == "None"))
         .count()
)
null_pct = round((null_count / row_count) * 100, 2)

print(f"\u2705 Saved: {PRE_SILVER_TABLE}")
print(f"   Total rows   : {row_count:,}")
print(f"   Null values  : {null_count:,}  ({null_pct}%)")


StatementMeta(, 061e5786-f0b0-4bad-8fe6-76fee4bd6201, 6, Finished, Available, Finished, False)


💾 Saving pre-silver availability table...
   Target: lh_Silver_StratusCoreTelecoms.dbo.pre_silver_stratus_availability
✅ Saved: lh_Silver_StratusCoreTelecoms.dbo.pre_silver_stratus_availability
   Total rows   : 4,800
   Null values  : 0  (0.0%)
